# ML - BOOSTING

In [55]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from utils1 import get_classifier_metrics
from sklearn.model_selection import GridSearchCV
from collections import Counter

## Paso 1. Lectura del conjunto de datos procesado

In [2]:
# Cargamos los dataframes 
with open('../data/processed/04_df_invalids_removed.pkl', 'rb') as f:
    df_invalids_removed = pickle.load(f)

with open('../data/processed/04_df_invalids_mode.pkl', 'rb') as f:
    df_invalids_mode = pickle.load(f)

with open('../data/processed/04_df_invalids_knn.pkl', 'rb') as f:
    df_invalids_knn = pickle.load(f)

## Paso 2. Split

In [5]:
X_invalids_removed = df_invalids_removed.drop('Outcome', axis= 1)
y_invalids_removed = df_invalids_removed['Outcome']

X_invalids_mode= df_invalids_mode.drop('Outcome', axis= 1)
y_invalids_mode = df_invalids_mode['Outcome']

X_invalids_knn= df_invalids_knn.drop('Outcome', axis= 1)
y_invalids_knn = df_invalids_knn['Outcome']

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_invalids_removed, y_invalids_removed, test_size=0.2, random_state=21)
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_invalids_mode, y_invalids_mode, test_size=0.2, random_state=21)
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_invalids_knn, y_invalids_knn, test_size=0.2, random_state=21)

## Paso 3. Modelado y Ajuste

In [7]:
model_boosting_default_1 = XGBClassifier(random_state=21)
model_boosting_default_1.fit(X_train_1, y_train_1)

model_boosting_default_2 = XGBClassifier(random_state=21)
model_boosting_default_2.fit(X_train_2, y_train_2)

model_boosting_default_3 = XGBClassifier(random_state=21)
model_boosting_default_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Paso 4. Predicción

In [9]:
y_pred_test_1 = model_boosting_default_1.predict(X_test_1)
y_pred_train_1 = model_boosting_default_1.predict(X_train_1)

y_pred_test_2 = model_boosting_default_2.predict(X_test_2)
y_pred_train_2 = model_boosting_default_2.predict(X_train_2)

y_pred_test_3 = model_boosting_default_3.predict(X_test_3)
y_pred_train_3 = model_boosting_default_3.predict(X_train_3)

In [12]:
default_model_metrics_1 = get_classifier_metrics(y_pred_test_1, y_test_1, y_pred_train_1, y_train_1, average='weighted')
default_model_metrics_1

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.734177,0.723725,0.726029,0.734177


In [13]:
default_model_metrics_2 = get_classifier_metrics(y_pred_test_2, y_test_2, y_pred_train_2, y_train_2, average='weighted')
default_model_metrics_2

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.701299,0.680869,0.699905,0.701299


In [14]:
default_model_metrics_3 = get_classifier_metrics(y_pred_test_3, y_test_3, y_pred_train_3, y_train_3, average='weighted')
default_model_metrics_3

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.701299,0.685789,0.696685,0.701299


### Modelo optimizado con los hiperparámetros con `GridSearchCV`

In [ ]:
'n_estimators'# número de árboles
    'max_depth' # profundidad máxima de cada árbol
    'learning_rate': # tasa de aprendizaje
    'subsample': # proporción de muestras usadas para entrenar cada árbol
    'colsample_bytree':  # proporción de características usadas por árbol
    'gamma': # regularización: requiere mejora mínima de pérdida para dividir
    'reg_alpha': # L1 regularization (sparse features)
    'reg_lambda': # L2 regularization (default is 1)

In [59]:
# Calcular manualmente el peso para balancear clases (0: no diabetes, 1: diabetes)

counts = Counter(y_train_3)
neg, pos = counts[0], counts[1]

scale_pos_weight_3 = neg / pos

In [60]:
param_grid = {'n_estimators': [20, 30, 40, 50], 
              'max_depth' : [3, 5, 7],
              'learning_rate': [0.01, 0.1, 0.2],          
              'subsample': [0.8, 1.0],                    
              'colsample_bytree': [0.8, 1.0],              
              'gamma': [0, 0.5, 1],
              'scale_pos_weight': [1, scale_pos_weight_3]}  

grid_1 = GridSearchCV(model_boosting_default_1,
                      param_grid,
                      scoring='recall',
                      cv=5)

grid_2 = GridSearchCV(model_boosting_default_2,
                      param_grid,
                      scoring='recall',
                      cv=5)

grid_3 = GridSearchCV(model_boosting_default_3,
                      param_grid,
                      scoring='recall',
                      cv=5)                                    

In [61]:
# Entrenamos el grid con los hiperparametros
grid_1.fit(X_train_1, y_train_1)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_1.best_params_

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x71d5a5410810>>
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/xgboost/core.py", line 585, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument

KeyboardInterrupt: 
/home/vscode/.local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
1 fits failed out of a total of 4320.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/sklearn/model_se

{'colsample_bytree': 1.0,
 'gamma': 0,
 'learning_rate': 0.01,
 'max_depth': 3,
 'n_estimators': 20,
 'scale_pos_weight': 1.9519230769230769,
 'subsample': 1.0}

In [45]:
grid_2.fit(X_train_2, y_train_2)
grid_2.best_params_

{'colsample_bytree': 0.8,
 'gamma': 0.5,
 'learning_rate': 0.2,
 'max_depth': 7,
 'n_estimators': 20,
 'subsample': 0.8}

In [ ]:
# Entrenamos el grid con los hiperparametros
grid_3.fit(X_train_3, y_train_3)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_3.best_params_

In [47]:
#Modelos con los mejores parametros
grid_boosting_1 = grid_1.best_estimator_
grid_boosting_2 = grid_2.best_estimator_
grid_boosting_3 = grid_3.best_estimator_

In [48]:
# Repetimos el entrenamiento pero ahora con el grid que tiene los hiperparametros establecidos
grid_boosting_1.fit(X_train_1, y_train_1)
grid_boosting_2.fit(X_train_2, y_train_2)
grid_boosting_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [49]:
y_pred_test_1_grid = grid_boosting_1.predict(X_test_1)
y_pred_train_1_grid = grid_boosting_1.predict(X_train_1)

y_pred_test_2_grid = grid_boosting_2.predict(X_test_2)
y_pred_train_2_grid = grid_boosting_2.predict(X_train_2)

y_pred_test_3_grid = grid_boosting_3.predict(X_test_3)
y_pred_train_3_grid = grid_boosting_3.predict(X_train_3)

In [50]:
grid_boosting_model_metrics_1 = get_classifier_metrics(y_pred_test_1_grid, y_test_1, y_pred_train_1_grid, y_train_1, average='weighted')
grid_boosting_model_metrics_1 


,Accuracy,F1 Score,Precision,Recall
Train set,0.949045,0.949045,0.949045,0.949045
Test set,0.746835,0.738678,0.739992,0.746835


In [51]:
grid_boosting_model_metrics_2 = get_classifier_metrics(y_pred_test_2_grid, y_test_2, y_pred_train_2_grid, y_train_2, average='weighted')
grid_boosting_model_metrics_2

,Accuracy,F1 Score,Precision,Recall
Train set,0.946254,0.945503,0.947319,0.946254
Test set,0.701299,0.690035,0.695159,0.701299


In [52]:
grid_boosting_model_metrics_3 = get_classifier_metrics(y_pred_test_3_grid, y_test_3, y_pred_train_3_grid, y_train_3, average='weighted')
grid_boosting_model_metrics_3

,Accuracy,F1 Score,Precision,Recall
Train set,0.978827,0.978709,0.979069,0.978827
Test set,0.733766,0.722834,0.731800,0.733766
